# MRI H5 Metadata Exploration

This notebook explores and summarizes metadata from MRI HDF5 (H5) files. It includes loading k-space data, parsing XML metadata, statistical analysis, and visualization of both metadata and k-space data.

In [1]:
# get home folder path
import os
import random
import ismrmrd
import h5py
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import xml.dom.minidom as minidom
import xml.dom.minidom
from collections import Counter
import torch

home_folder = os.path.expanduser("~")
parent_folder = os.path.abspath(os.path.join(home_folder, "..", ".."))
# cd to data folder
os.chdir(parent_folder)

data_folder = Path(parent_folder) / "data/datasets/msk_mri_h5/h5"

h5_paths = sorted(
    p for p in data_folder.iterdir()
    if p.suffix.lower() in {'.h5', '.hdf5'} and p.is_file()
)
print(f'Found {len(h5_paths)} H5 files in: {data_folder}')

# choose random file from folder
random_file = random.choice(h5_paths)
random_file = "clean_meas_MID00017_FID135133_COR_T2.h5"  # specific file for testing
print(f"Random file chosen from the data folder: {random_file}")


Found 713 H5 files in: /data/datasets/msk_mri_h5/h5
Random file chosen from the data folder: clean_meas_MID00017_FID135133_COR_T2.h5


## Load K-space and Print Metadata Function

Defines a function to load k-space data from an HDF5 file using ISMRMRD, optionally printing the XML metadata structure. Loads k-space data for the selected file.

In [3]:
def load_h5_kspace_and_print_metadata(path, print_metadata=False):
    """
    Load k-space from an HDF5 file and print XML metadata.
    Returns:
      kspace: np.ndarray with shape (S, C, H, W, 2) and dtype float32
    """
    # Load k-space using ismrmrd
    dset = ismrmrd.Dataset(path, 'dataset', create_if_needed=False)
    n = dset.number_of_acquisitions()
    if n == 0:
        raise ValueError('No acquisitions found in ISMRMRD file.')
    acq0 = dset.read_acquisition(0)
    nSamp = acq0.number_of_samples
    nCh = acq0.active_channels
    data = np.empty((n, nCh, nSamp), dtype=np.complex64)
    ky = np.empty(n, dtype=np.int32)
    kz = np.empty(n, dtype=np.int32)
    sl = np.empty(n, dtype=np.int32)
    for i in range(n):
        acq = dset.read_acquisition(i)
        data[i] = acq.data.astype(np.complex64)
        md = acq.idx
        ky[i] = getattr(md, 'kspace_encode_step_1', 0)
        kz[i] = getattr(md, 'kspace_encode_step_2', 0)
        sl[i] = getattr(md, 'slice', 0)
    dset.close()

    Ny = int(ky.max() + 1) if ky.max() >= 0 else n
    Nz = int(kz.max() + 1) if kz.max() >= 0 else 1
    Nslices = int(sl.max() + 1) if sl.max() >= 0 else 1
    Nx = nSamp
    S = Nslices * Nz
    kspace = np.zeros((S, nCh, Ny, Nx), dtype=np.complex64)
    slice_map = {}
    s_idx = 0
    for s in range(Nslices):
        for z in range(Nz):
            slice_map[(s, z)] = s_idx
            s_idx += 1
    order = np.lexsort((ky, kz, sl))
    for i in order:
        s = int(sl[i]); z = int(kz[i]); y = int(ky[i])
        si = slice_map[(s, z)]
        if 0 <= y < Ny:
            kspace[si, :, y, :] = data[i]
    ks_ri = np.stack((kspace.real, kspace.imag), axis=-1).astype(np.float32)
    #print(f"K-space shape: {ks_ri.shape}, dtype: {ks_ri.dtype}")

    if print_metadata:
        with h5py.File(path, "r") as f2:
            #print all keys in the file
            print("HDF5 file keys:")
            def print_h5_keys(name, obj):
                print(name)
            f2.visititems(print_h5_keys)
            xml_bytes = f2["dataset"]["xml"][0][:]
            xml_str = xml_bytes.decode('utf-8') if isinstance(xml_bytes, (bytes, bytearray)) else str(xml_bytes)
            dom = xml.dom.minidom.parseString(xml_str)

            def print_elements(node, indent=0):
                for child in node.childNodes:
                    if child.nodeType == child.ELEMENT_NODE:
                        print("  " * indent + f"{child.tagName}:", end=" ")
                        text = ""
                        if child.firstChild and child.firstChild.nodeType == child.TEXT_NODE:
                            text = child.firstChild.data.strip()
                        if text:
                            print(text)
                        else:
                            print()
                        print_elements(child, indent + 1)

            print_elements(dom.documentElement)

    return ks_ri

kspace_np = load_h5_kspace_and_print_metadata(os.path.join(data_folder, random_file), print_metadata=True)

HDF5 file keys:
dataset
dataset/data
dataset/xml
dataset/xml_backup_before_shape_fix
subjectInformation: 
  patientName: xxxxxxxxxxxxxxxxxxxxx
  patientWeight_kg: 81.6470032
  patientHeight_m: 1803
  patientID: 18.0.261379110
  patientGender: F
studyInformation: 
  studyTime: 06:25:28
  studyID: 18.0.261379118
measurementInformation: 
  measurementID: 169597_261379110_261379118_17
  patientPosition: HFS
  protocolName: COR T2
  frameOfReferenceUID: 1.3.12.2.1107.5.2.41.169597.1.20250923062309749.0.0.5010
acquisitionSystemInformation: 
  systemVendor: SIEMENS
  systemModel: Avanto_fit
  systemFieldStrength_T: 1.49399996
  relativeReceiverNoiseBandwidth: 0.792999983
  receiverChannels: 12
  institutionName: UUOC RAD MR
experimentalConditions: 
  H1resonanceFrequency_Hz: 63675433
encoding: 
  encodedSpace: 
    matrixSize: 
      x: 768
      y: 732
      z: 1
    fieldOfView_mm: 
      x: 560
      y: 533.119995
      z: 6
  reconSpace: 
    matrixSize: 
      x: 384
      y: 384
      z

## Metadata Statistics

Extracts XML metadata from H5 files, parse relevant fields, and aggregate metadata across all files. Computes and visualizes statistics for key metadata fields.

In [ ]:
# ------------------- HELPERS -------------------
def _first_text_from_tag(parent_node, tag_name):
    nodes = parent_node.getElementsByTagName(tag_name)
    if nodes:
        for node in nodes[0].childNodes:
            if node.nodeType == node.TEXT_NODE:
                text = node.data.strip()
                if text:
                    return text
    return None

def read_xml_from_h5(h5_path: Path):
    try:
        with h5py.File(h5_path, 'r') as f:
            dataset = f.get('dataset')
            if dataset is not None and 'xml' in dataset:
                raw = dataset['xml'][0]
                if isinstance(raw, (bytes, bytearray)):
                    return raw.decode('utf-8', errors='ignore')
                return str(raw)
    except Exception:
        pass
    return None

def parse_xml_fields(xml_str: str):
    parsed = {
        'patientID': None,
        'tStudyDescription': None,
        'patientPosition': None,
        'receiverChannels': None,
        'Nx': None,
        'Ny': None,
        'slices': None,
    }
    try:
        doc = minidom.parseString(xml_str)
    except Exception:
        return parsed

    subj_nodes = doc.getElementsByTagName('subjectInformation')
    if subj_nodes:
        parsed['patientID'] = _first_text_from_tag(subj_nodes[0], 'patientID')

    for up in doc.getElementsByTagName('userParameterString'):
        name = _first_text_from_tag(up, 'name')
        value = _first_text_from_tag(up, 'value')
        if name == 'tStudyDescription':
            parsed['tStudyDescription'] = value
        elif name == 'Nx':
            try:
                parsed['Nx'] = int(value)
            except (TypeError, ValueError):
                pass
        elif name == 'Ny':
            try:
                parsed['Ny'] = int(value)
            except (TypeError, ValueError):
                pass

    meas = doc.getElementsByTagName('measurementInformation')
    if meas:
        parsed['patientPosition'] = _first_text_from_tag(meas[0], 'patientPosition')

    acq = doc.getElementsByTagName('acquisitionSystemInformation')
    if acq:
        rc = _first_text_from_tag(acq[0], 'receiverChannels')
        if rc:
            try:
                parsed['receiverChannels'] = int(rc)
            except ValueError:
                pass

    slices = doc.getElementsByTagName('slice')
    if slices:
        maximum = _first_text_from_tag(slices[0], 'maximum')
        try:
            parsed['slices'] = int(maximum) + 1 if maximum is not None else None
        except (TypeError, ValueError):
            pass

    return parsed

# ------------------- PARSE ALL FILES -------------------
rows, errors = [], []
for path in h5_paths:
    xml_str = read_xml_from_h5(path)
    if not xml_str:
        errors.append((path.name, 'No XML'))
        continue
    rows.append(parse_xml_fields(xml_str))

# ------------------- STATISTICS -------------------
patient_ids = [r['patientID'] for r in rows if r['patientID']]
t_study_descs = [r['tStudyDescription'] for r in rows if r['tStudyDescription']]
patient_positions = [r['patientPosition'] for r in rows if r['patientPosition']]

receiver_channels = [r['receiverChannels'] for r in rows if r['receiverChannels'] is not None]
Nx_list = [r['Nx'] for r in rows if r['Nx'] is not None]
Ny_list = [r['Ny'] for r in rows if r['Ny'] is not None]
slices = [r['slices'] for r in rows if r['slices'] is not None]

xy_data, xy_labels = [], []
if Nx_list:
    xy_data.append(np.asarray(Nx_list, dtype=float))
    xy_labels.append('Nx')
if Ny_list:
    xy_data.append(np.asarray(Ny_list, dtype=float))
    xy_labels.append('Ny')

if xy_data:
    with plt.rc_context({
        'font.family': 'serif',
        'axes.spines.top': False,
        'axes.spines.right': False,
    }):
        fig, ax = plt.subplots(figsize=(6, 5), dpi=100)
        box = ax.boxplot(
            xy_data,
            labels=xy_labels,
            widths=0.45,
            showmeans=True,
            meanprops=dict(marker='D', markeredgecolor='#333333',
                           markerfacecolor='#55A868', markersize=6),
            boxprops=dict(linewidth=1.2, facecolor='#C1D4F4', color='#C1D4F4'),
            whiskerprops=dict(color='#4C72B0', linewidth=1.2),
            capprops=dict(color='#4C72B0', linewidth=1.2),
            medianprops=dict(color='#DD8452', linewidth=2.0),
            flierprops=dict(marker='o', markerfacecolor='#222222', markersize=4, linestyle='none', alpha=0.5),
            patch_artist=True,
        )
        # Remove black borders from boxes
        for patch in box['boxes']:
            patch.set_edgecolor('#4C72B0')
            patch.set_linewidth(1.2)
        ax.set_ylabel('Matrix size (pixels)', fontsize=13)
        ax.set_xlabel('Dimension', fontsize=13)
        ax.set_title('Kspace Sizes', fontsize=17, fontweight='bold', pad=12)
        ax.yaxis.set_major_locator(mticker.MaxNLocator(nbins='auto'))
        ax.grid(axis='y', color='#D0D0D0', linestyle='--', linewidth=0.7, alpha=0.8)
        fig.tight_layout()
        plt.show()
else:
    print('No Nx/Ny data available for plotting.')

def summarize(arr, name):
    if not arr:
        return
    arr = np.asarray(arr, dtype=float)
    std = arr.std(ddof=1) if arr.size > 1 else 0.0
    summary_line = (
        '{name}: min={min:.0f}, median={median:.0f}, '
        'mean={mean:.2f}, std={std:.2f}, max={max:.0f}'
    ).format(
        name=name,
        n=arr.size,
        min=arr.min(),
        median=np.median(arr),
        mean=arr.mean(),
        std=std,
        max=arr.max(),
    )
    print(summary_line)

print('\n=== Summary ===')
print(f'Processed files: {len(h5_paths)} | Parsed: {len(rows)} | Errors: {len(errors)}')
print(f'Unique patients: {len(set(patient_ids))}')
print(f'Unique tStudyDescription: {len(set(t_study_descs))}')
print(f'Unique patient positions: {len(set(patient_positions))}')

summarize(receiver_channels, 'Receiver channels')
summarize(Nx_list, 'Nx')
summarize(Ny_list, 'Ny')
summarize(slices, 'Slices')

if t_study_descs:
    print('\nTop tStudyDescription by file count:')
    for name, count in Counter(t_study_descs).most_common(5):
        print(f'  {name}: {count}')

if patient_positions:
    print('\nPatient positions by file count:')
    for pos, count in Counter(patient_positions).most_common():
        print(f'  {pos}: {count}')


## File Size Statistics and Histogram

In [ ]:
file_sizes_mb = np.array([p.stat().st_size / (1024 * 1024) for p in h5_paths], dtype=float)

if file_sizes_mb.size:
    mean_mb = float(file_sizes_mb.mean())
    median_mb = float(np.median(file_sizes_mb))
    std_mb = float(file_sizes_mb.std(ddof=1)) if file_sizes_mb.size > 1 else 0.0

    with plt.rc_context({
        'font.family': 'serif',
        'axes.spines.top': False,
        'axes.spines.right': False,
    }):
        fig, ax = plt.subplots(figsize=(8, 5))
        ax.hist(
            file_sizes_mb,
            bins='auto',
            color='#4C72B0',
            edgecolor='white',
            alpha=0.85
        )

        ax.axvline(median_mb, color='#DD8452', linewidth=1.6, linestyle='--', label=f'Median = {median_mb:.2f} MB')
        ax.set_title('Distribution of File Sizes', fontsize=16, fontweight='bold')
        ax.set_xlabel('File size (MB)', fontsize=13)
        ax.set_ylabel('Frequency', fontsize=13)

        ax.yaxis.set_major_locator(mticker.MaxNLocator(integer=True))
        ax.grid(axis='y', color='#CCCCCC', linestyle='--', linewidth=0.6, alpha=0.7)
        ax.set_axisbelow(True)

        ax.text(
            0.98,
            0.95,
            f'Mean = {mean_mb:.2f} MB\nSD = {std_mb:.2f} MB',
            transform=ax.transAxes,
            ha='right',
            va='top',
            fontsize=11,
            bbox=dict(boxstyle='round', facecolor='white', edgecolor='#B0B0B0', alpha=0.85)
        )

        ax.legend(frameon=False, fontsize=11)
        fig.tight_layout()
        plt.show()
else:
    print('No file sizes found for histogram.')


## Torch-based K-space and Image Visualization

Converts loaded k-space data to PyTorch tensors and visualizes both the reconstructed image and k-space magnitude for each slice using iFFT and root-sum-of-squares coil combination.

In [ ]:
def plot_kspace_and_recon(kspace_np: torch.Tensor, slice_idx: int):
    # Expect shape (S, C, H, W, 2) from load_h5_kspace
    kspace = torch.from_numpy(kspace_np).contiguous()
    assert kspace.ndim == 5 and kspace.shape[-1] == 2, f"Expected (S, C, H, W, 2), got {tuple(kspace.shape)}"

    S, Cc, H, W, _ = kspace.shape

    def ifft2c_torch(x_ri: torch.Tensor) -> torch.Tensor:
        """
        Centered 2D iFFT for RI-paired tensors.
        Input: (..., H, W, 2) with last dim [real, imag].
        Returns: same shape (..., H, W, 2).
        """
        xc = torch.view_as_complex(x_ri.to(torch.float32))  # (..., H, W) complex64
        imgc = torch.fft.ifftshift(
            torch.fft.ifft2(torch.fft.fftshift(xc, dim=(-2, -1)), norm="ortho"),
            dim=(-2, -1),
        )
        return torch.view_as_real(imgc)  # (..., H, W, 2)


    def rss_from_ri(x_ri: torch.Tensor, coil_dim: int = 0) -> torch.Tensor:
        """Root-sum-of-squares coil combine for RI-paired tensors with a coil dimension.
        Input: (C, H, W, 2)  -> Output: (H, W)
        """
        mag2 = x_ri[..., 0] ** 2 + x_ri[..., 1] ** 2  # (C, H, W)
        return torch.sqrt(mag2.sum(dim=coil_dim) + 1e-12)


    with torch.no_grad():
        for s in range(S):
            ks_slice = kspace[s]  # (C, H, W, 2)

            # Reconstruct complex image via centered iFFT, then coil-combine with RSS
            img_ri = ifft2c_torch(ks_slice)              # (C, H, W, 2)
            img_mag = rss_from_ri(img_ri, coil_dim=0)    # (H, W)

            # k-space magnitude for display (log), collapse coils by max
            ks_c = torch.view_as_complex(ks_slice)
            ks_mag_log = torch.log1p(ks_c.abs()).amax(dim=0)  # (H, W)

            fig, axs = plt.subplots(1, 2, figsize=(10, 5), dpi=160)
            axs[0].imshow(img_mag.cpu().numpy(), cmap='gray')
            axs[0].set_title(f'Reconstructed Image - Slice {s+1}')
            axs[0].axis('off')
            axs[1].imshow(ks_mag_log.cpu().numpy(), cmap='gray')
            axs[1].set_title(f'k-space Magnitude (log) - Slice {s+1}')
            axs[1].axis('off')
            plt.show()

In [13]:
kspace_np = load_h5_kspace_and_print_metadata(os.path.join(data_folder, random_file), print_metadata=False)
plot_kspace_and_recon(kspace_np, slice_idx=0)                                              